# BIM Generation Pipeline

## What This Notebook Does

This notebook generates **synthetic BIM-style renders** from real construction site
photographs using **ControlNet + Stable Diffusion 1.5**.

The goal is to produce a paired `(real_photo, BIM_render)` dataset that will be used
to fine-tune a Vision-Language Model (VLM) for automated construction progress estimation.

Since paired real/BIM data does not exist in the wild, we generate the BIM side synthetically
by conditioning SD 1.5 on depth maps extracted from the real photos.

```
Real Photo  -->  MiDaS Depth Map  -->  ControlNet (SD 1.5)  -->  BIM-style Render
                                                                        |
                              Paired Dataset  -->  VLM Fine-tuning  -->  Progress Estimation
```

## Dataset

**LouisChen15/ConstructionSite** — 10,013 labeled construction site images with metadata:
- `view`: elevation / plan / perspective / aerial
- `camera_distance`: close / mid / far
- `image_caption`, bounding boxes, safety annotations

## Key Results (50-sample test)

| Metric | Value |
|---|---|
| BIM Style Score | **0.61** (target > 0.55) |
| Condition mode | depth (MiDaS) |
| Guidance scale | 12.0 |
| ControlNet scale | 1.2 |
| Inference steps | 35 |

## Compatibility Notes

This Colab environment ships with **PyTorch 2.10/2.12 nightly (cu128)** which is
incompatible with the diffusers/torchvision ecosystem. Two fixes are applied:

1. **PyTorch downgrade** to 2.5.1+cu121 (cu121 wheels run fine on cu128 hardware)
2. **JIT patch** — `torch.jit.script` is wrapped in a safe fallback before diffusers
   is imported, preventing the `JITCallable._set_src()` crash

## Execution Order

| Cell | Action | Notes |
|---|---|---|
| 1 | Install PyTorch 2.5.1 | **Restart session after this** |
| 2 | Verify environment | Check versions and nms test |
| 3 | Mount Google Drive | Outputs saved here |
| 4 | GitHub setup | Clone/pull repo |
| 5 | Install dependencies | diffusers, controlnet-aux, etc. |
| 6 | Load dataset | HuggingFace login + ConstructionSite |
| 7 | Write source files | prompt_factory, condition_generator, bim_generator |
| 8 | Load models | JIT patch + MiDaS + ControlNet + SD 1.5 |
| 9 | Single sample test | Verify pipeline end-to-end |
| 10 | Full generation | Process all samples, save to Drive |
| 11 | Visualize results | Random grid of real/depth/BIM |
| 12 | BIM style score | Quality metric |
| 13 | Push to GitHub | Commit source files + notebook |

> **Colab Secrets required** (left panel, key icon):
> `HF_TOKEN`, `GITHUB_TOKEN`, `GITHUB_USERNAME`, `GITHUB_REPO`, `GIT_EMAIL`, `GIT_NAME`

## Cell 1 — Install PyTorch 2.5.1

> Run this cell first, then **Runtime -> Restart session**. Continue from Cell 2.

In [ ]:
# Colab ships with PyTorch 2.10/2.12 nightly (cu128).
# diffusers 0.36 + torchvision require PyTorch 2.5.1 stable.
# cu121 wheels are backward-compatible with cu128 hardware.

import subprocess

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout: print(r.stdout[-400:])
    if r.returncode != 0 and r.stderr: print('STDERR:', r.stderr[-200:])
    return r.returncode

print('Removing existing torch ecosystem...')
run('pip uninstall -y torch torchvision torchaudio xformers 2>/dev/null')
run('pip cache purge')

print('Installing PyTorch 2.5.1+cu121...')
ret = run(
    'pip install torch==2.5.1+cu121 torchvision==0.20.1+cu121 torchaudio==2.5.1+cu121 '
    '--index-url https://download.pytorch.org/whl/cu121 '
    '--force-reinstall --no-deps -q'
)

if ret == 0:
    print('\n[OK] PyTorch 2.5.1 installed.')
    print('=' * 50)
    print('  Now go to Runtime -> Restart session')
    print('  Then continue from Cell 2.')
    print('=' * 50)
else:
    print('[ERROR] Installation failed — check logs above')


## Cell 2 — Verify Environment

Run after restarting. Re-run Cell 1 if any check fails.

In [ ]:
import torch, torchvision

print(f'PyTorch     : {torch.__version__}')        # expected: 2.5.1+cu121
print(f'torchvision : {torchvision.__version__}')  # expected: 0.20.1+cu121
print(f'CUDA        : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU         : {torch.cuda.get_device_name(0)}')
    print(f'VRAM        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# torchvision::nms must pass for ControlNet to work
try:
    boxes  = torch.tensor([[0,0,1,1],[0,0,.9,.9]], dtype=torch.float).cuda()
    scores = torch.tensor([0.9, 0.8]).cuda()
    torchvision.ops.nms(boxes, scores, 0.5)
    print('torchvision nms : OK')
except Exception as e:
    print(f'torchvision nms : FAILED -> {e}')
    print('  Re-run Cell 1 and restart session')


## Cell 3 — Mount Google Drive

All generated images are saved to Drive to persist across sessions.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/BIM_Generation'
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f'Project folder: {PROJECT_DIR}')


## Cell 4 — GitHub Setup

Add the following via **left panel -> key icon -> Add new secret**:

| Secret | Example |
|---|---|
| `GITHUB_TOKEN` | Personal Access Token (repo scope) |
| `GITHUB_USERNAME` | your-username |
| `GITHUB_REPO` | bim-generation-pipeline |
| `GIT_EMAIL` | you@email.com |
| `GIT_NAME` | Your Name |

In [ ]:
import os
from google.colab import userdata

try:
    GITHUB_TOKEN    = userdata.get('GITHUB_TOKEN')
    GITHUB_USERNAME = userdata.get('GITHUB_USERNAME')
    GITHUB_REPO     = userdata.get('GITHUB_REPO')
    GIT_EMAIL       = userdata.get('GIT_EMAIL')
    GIT_NAME        = userdata.get('GIT_NAME')
    print('Credentials loaded from Secrets')
except Exception:
    GITHUB_TOKEN    = input('GitHub Token: ').strip()
    GITHUB_USERNAME = input('GitHub Username: ').strip()
    GITHUB_REPO     = input('Repo name (e.g. bim-generation-pipeline): ').strip()
    GIT_EMAIL       = input('Email: ').strip()
    GIT_NAME        = input('Name: ').strip()

REPO_URL = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git'
REPO_DIR = f'/content/{GITHUB_REPO}'

os.system(f'git config --global user.email "{GIT_EMAIL}"')
os.system(f'git config --global user.name  "{GIT_NAME}"')

if not os.path.exists(REPO_DIR):
    ret = os.system(f'git clone {REPO_URL} {REPO_DIR}')
    print(f'Repo cloned: {REPO_DIR}' if ret == 0 else 'Clone failed — check token and repo name')
else:
    os.system(f'cd {REPO_DIR} && git pull origin main')
    print(f'Repo updated: {REPO_DIR}')

for d in ['src', 'samples', 'notebooks']:
    os.makedirs(os.path.join(REPO_DIR, d), exist_ok=True)

with open(os.path.join(REPO_DIR, '.gitignore'), 'w') as f:
    f.write('outputs/\n*.jpg\n*.png\n*.pt\n*.bin\n__pycache__/\n*.pyc\n.env\n')

print(f'REPO_DIR = {REPO_DIR}')


## Cell 5 — Install Dependencies

In [ ]:
import subprocess

def pip(cmd):
    r = subprocess.run(f'pip install -q {cmd}', shell=True, capture_output=True, text=True)
    print('  OK' if r.returncode == 0 else f'  FAILED: {r.stderr[-150:]}')

# diffusers 0.36.0 is required — 0.27.x has a broken single_file_utils import
print('diffusers 0.36.0...')
pip('diffusers==0.36.0 --force-reinstall --no-deps')

print('transformers + accelerate...')
pip('transformers>=4.40.0 accelerate')

print('controlnet-aux...')
pip('controlnet-aux==0.0.7')

print('xformers...')
pip('xformers==0.0.28.post3 --index-url https://download.pytorch.org/whl/cu121')

print('other packages...')
pip('datasets huggingface_hub opencv-python-headless scikit-image')

import importlib
print('\nVerification:')
for pkg in ['diffusers', 'transformers', 'datasets', 'cv2']:
    try:
        m = importlib.import_module(pkg)
        print(f'  {pkg}: {getattr(m, "__version__", "ok")}')
    except ImportError as e:
        print(f'  MISSING {pkg}: {e}')


## Cell 6 — HuggingFace Login & Load Dataset

Add `HF_TOKEN` to Colab Secrets (left panel, key icon).

Set `MAX_SAMPLES = None` to process the full 7K training set.

In [ ]:
from huggingface_hub import login
from google.colab import userdata
from datasets import load_dataset
import matplotlib.pyplot as plt

try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    print('HF token loaded from Secrets')
except Exception:
    HF_TOKEN = input('HuggingFace token: ').strip()

login(token=HF_TOKEN, add_to_git_credential=False)
print('HuggingFace login OK')

# ── Settings ─────────────────────────────────────
SPLIT       = 'train'
MAX_SAMPLES = 50     # set to None for the full 7K dataset
IMAGE_SIZE  = 512
# ─────────────────────────────────────────────────

print(f'Loading dataset (split={SPLIT}, max={MAX_SAMPLES})...')
ds = load_dataset('LouisChen15/ConstructionSite', split=SPLIT, token=HF_TOKEN)
if MAX_SAMPLES:
    ds = ds.select(range(min(MAX_SAMPLES, len(ds))))
print(f'Loaded {len(ds)} samples | columns: {ds.column_names}')

# Preview
n = min(3, len(ds))
fig, axes = plt.subplots(1, n, figsize=(5 * n, 4))
if n == 1: axes = [axes]
for i in range(n):
    s = ds[i]
    axes[i].imshow(s['image'])
    axes[i].set_title(f"{s.get('view','')}\n{s.get('camera_distance','')}", fontsize=8)
    axes[i].axis('off')
plt.suptitle('Dataset Preview', fontsize=12)
plt.tight_layout()
plt.show()


## Cell 7 — Write Source Files

Creates three modules in `src/`:

- **`prompt_factory.py`** — converts dataset metadata into ControlNet prompts (token-safe, < 77 tokens)
- **`condition_generator.py`** — generates MiDaS depth / HED edge / Canny maps
- **`bim_generator.py`** — ControlNet + SD 1.5 pipeline with JIT compatibility patch

In [ ]:
import os

if 'REPO_DIR' not in dir():
    from google.colab import userdata
    try:    GITHUB_REPO = userdata.get('GITHUB_REPO')
    except: GITHUB_REPO = input('Repo name: ').strip()
    REPO_DIR = f'/content/{GITHUB_REPO}'

SRC = os.path.join(REPO_DIR, 'src')
os.makedirs(SRC, exist_ok=True)

# ── prompt_factory.py ─────────────────────────────────────────────
# Prompts are kept under the CLIP 77-token limit.
# build_prompt() maps dataset view/distance fields to positive/negative prompts.
with open(os.path.join(SRC, 'prompt_factory.py'), 'w') as f:
    f.write(
'VIEW_PROMPTS = {\n'
'    "elevation view":   "front elevation view",\n'
'    "plan view":        "top-down plan view",\n'
'    "perspective view": "perspective view",\n'
'    "aerial view":      "aerial perspective view",\n'
'    "close-up view":    "close-up detail view",\n'
'    "interior view":    "interior section view",\n'
'}\n'
'DISTANCE_HINTS = {\n'
'    "close distance": "detailed close-up",\n'
'    "mid distance":   "medium distance",\n'
'    "far distance":   "wide-angle overview",\n'
'}\n'
'\n'
'def build_prompt(sample: dict) -> tuple:\n'
'    """Returns (positive, negative) prompt for a dataset sample."""\n'
'    view     = VIEW_PROMPTS.get(sample.get("view", ""), "perspective view")\n'
'    distance = DISTANCE_HINTS.get(sample.get("camera_distance", ""), "medium distance")\n'
'    positive = (\n'
'        f"BIM render, {view}, {distance}, "\n'
'        f"3D building model, flat colors, structural elements, "\n'
'        f"Revit style, no texture, technical CAD drawing, "\n'
'        f"concrete steel structure, clean background"\n'
'    )\n'
'    negative = (\n'
'        "(workers:2.0), (people:2.0), (humans:2.0), "\n'
'        "(hard hat:1.5), (watermark:2.0), (text:2.0), "\n'
'        "photorealistic, photo, dirt, vegetation, "\n'
'        "shadows, blur, low quality"\n'
'    )\n'
'    return positive, negative\n'
    )
print('prompt_factory.py written')

# ── condition_generator.py ────────────────────────────────────────
# Supported modes: depth | hed | canny | combined
with open(os.path.join(SRC, 'condition_generator.py'), 'w') as f:
    f.write(
'import cv2\n'
'import numpy as np\n'
'from PIL import Image\n'
'from controlnet_aux import MidasDetector, HEDdetector\n'
'\n'
'class ConditionGenerator:\n'
'    def __init__(self, mode: str = "depth"):\n'
'        self.mode = mode\n'
'        print(f"Condition mode: {mode}")\n'
'        if mode in ("depth", "combined"):\n'
'            self.depth = MidasDetector.from_pretrained("lllyasviel/Annotators")\n'
'            print("  MiDaS loaded")\n'
'        if mode in ("hed", "combined"):\n'
'            self.hed = HEDdetector.from_pretrained("lllyasviel/Annotators")\n'
'            print("  HED loaded")\n'
'        print("ConditionGenerator ready")\n'
'\n'
'    def generate(self, image: Image.Image) -> Image.Image:\n'
'        img = image.convert("RGB")\n'
'        if self.mode == "depth":    return self.depth(img).convert("RGB")\n'
'        if self.mode == "hed":      return self.hed(img).convert("RGB")\n'
'        if self.mode == "canny":    return self._canny(img)\n'
'        if self.mode == "combined": return self._combined(img)\n'
'        raise ValueError(f"Unknown mode: {self.mode}")\n'
'\n'
'    def _canny(self, img):\n'
'        arr   = np.array(img)\n'
'        gray  = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)\n'
'        edges = cv2.Canny(gray, 100, 200)\n'
'        return Image.fromarray(cv2.cvtColor(edges, cv2.COLOR_GRAY2RGB))\n'
'\n'
'    def _combined(self, img):\n'
'        # 60% depth + 40% HED\n'
'        d = np.array(self.depth(img).convert("L"))\n'
'        h = np.array(self.hed(img).convert("L"))\n'
'        c = (d * 0.6 + h * 0.4).astype(np.uint8)\n'
'        return Image.fromarray(c).convert("RGB")\n'
    )
print('condition_generator.py written')

# ── bim_generator.py ──────────────────────────────────────────────
# JIT patch is embedded here so the module is self-contained.
# PyTorch 2.5.1 + diffusers 0.36 have a JITCallable._set_src() conflict;
# wrapping torch.jit.script in a safe fallback resolves it.
with open(os.path.join(SRC, 'bim_generator.py'), 'w') as f:
    f.write(
'import os\n'
'os.environ["PYTORCH_JIT"] = "0"\n'
'\n'
'import torch, torch.jit\n'
'\n'
'# JIT compatibility patch\n'
'_orig_script = torch.jit.script\n'
'def _safe_script(fn, *args, **kwargs):\n'
'    try:    return _orig_script(fn, *args, **kwargs)\n'
'    except: return fn\n'
'torch.jit.script = _safe_script\n'
'try:\n'
'    from torch.jit._script import ScriptMeta\n'
'    _orig_init = ScriptMeta.__init__\n'
'    def _patched_init(cls, name, bases, attrs):\n'
'        try:    return _orig_init(cls, name, bases, attrs)\n'
'        except TypeError: pass\n'
'    ScriptMeta.__init__ = _patched_init\n'
'except Exception: pass\n'
'\n'
'from PIL import Image\n'
'from diffusers import (\n'
'    ControlNetModel,\n'
'    StableDiffusionControlNetPipeline,\n'
'    UniPCMultistepScheduler,\n'
')\n'
'\n'
'CONTROLNET_IDS = {\n'
'    "depth":    "lllyasviel/sd-controlnet-depth",\n'
'    "hed":      "lllyasviel/sd-controlnet-hed",\n'
'    "canny":    "lllyasviel/sd-controlnet-canny",\n'
'    "combined": "lllyasviel/sd-controlnet-depth",\n'
'}\n'
'\n'
'class BIMGenerator:\n'
'    def __init__(self, condition_mode: str = "depth", seed: int = 42):\n'
'        self.device = "cuda" if torch.cuda.is_available() else "cpu"\n'
'        self.gen    = torch.Generator(self.device).manual_seed(seed)\n'
'        print(f"Loading ControlNet: {CONTROLNET_IDS[condition_mode]}")\n'
'        controlnet = ControlNetModel.from_pretrained(\n'
'            CONTROLNET_IDS[condition_mode], torch_dtype=torch.float16)\n'
'        print("Loading SD 1.5...")\n'
'        self.pipe = StableDiffusionControlNetPipeline.from_pretrained(\n'
'            "runwayml/stable-diffusion-v1-5",\n'
'            controlnet=controlnet, torch_dtype=torch.float16,\n'
'            safety_checker=None, requires_safety_checker=False,\n'
'        )\n'
'        self.pipe.scheduler = UniPCMultistepScheduler.from_config(\n'
'            self.pipe.scheduler.config)\n'
'        self.pipe = self.pipe.to(self.device)\n'
'        print(f"BIMGenerator ready | device: {self.device}")\n'
'\n'
'    def generate(self, condition_map, prompt, negative_prompt,\n'
'                 steps=35, guidance=12.0, cn_scale=1.2, size=512):\n'
'        cond = condition_map.resize((size, size))\n'
'        with torch.inference_mode():\n'
'            out = self.pipe(\n'
'                prompt=prompt, negative_prompt=negative_prompt, image=cond,\n'
'                num_inference_steps=steps, guidance_scale=guidance,\n'
'                controlnet_conditioning_scale=cn_scale,\n'
'                generator=self.gen, width=size, height=size,\n'
'            )\n'
'        return out.images[0]\n'
    )
print('bim_generator.py written')
print(f'Source files: {os.listdir(SRC)}')


## Cell 8 — Apply JIT Patch & Load Models

Run this cell **after every session restart** — it re-applies the JIT patch,
restores all pipeline variables, and loads models into GPU memory.
First run takes ~5-10 minutes (downloads from HuggingFace).

In [ ]:
import os, sys

# JIT patch must be applied BEFORE any diffusers import
os.environ['PYTORCH_JIT'] = '0'
import torch, torch.jit

_orig_script = torch.jit.script
def _safe_script(fn, *args, **kwargs):
    try:    return _orig_script(fn, *args, **kwargs)
    except: return fn
torch.jit.script = _safe_script

try:
    from torch.jit._script import ScriptMeta
    _orig_init = ScriptMeta.__init__
    def _patched_init(cls, name, bases, attrs):
        try:    return _orig_init(cls, name, bases, attrs)
        except TypeError: pass
    ScriptMeta.__init__ = _patched_init
except Exception:
    pass

print(f'JIT patch applied | torch: {torch.__version__}')

# Recover variables if session was restarted
if 'REPO_DIR' not in dir():
    from google.colab import userdata
    try:    GITHUB_REPO = userdata.get('GITHUB_REPO')
    except: GITHUB_REPO = input('Repo name: ').strip()
    REPO_DIR = f'/content/{GITHUB_REPO}'

if 'PROJECT_DIR' not in dir():
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/BIM_Generation'

# ── Pipeline settings ─────────────────────────────────────────────
CONDITION_MODE = 'depth'   # depth | hed | canny | combined
STEPS          = 35
GUIDANCE       = 12.0
CN_SCALE       = 1.2
SIZE           = 512

# ── Output directories ────────────────────────────────────────────
OUT_BASE = os.path.join(PROJECT_DIR, 'outputs', CONDITION_MODE)
REAL_DIR = os.path.join(OUT_BASE, 'real')
BIM_DIR  = os.path.join(OUT_BASE, 'bim')
COND_DIR = os.path.join(OUT_BASE, 'condition_maps')
for d in [REAL_DIR, BIM_DIR, COND_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Output: {OUT_BASE}')

# ── Import modules ────────────────────────────────────────────────
for mod in ['prompt_factory', 'condition_generator', 'bim_generator']:
    sys.modules.pop(mod, None)

sys.path.insert(0, os.path.join(REPO_DIR, 'src'))

from prompt_factory      import build_prompt
from condition_generator import ConditionGenerator
from bim_generator       import BIMGenerator

print('Modules imported')

# ── Load models ───────────────────────────────────────────────────
print('\nLoading condition model (MiDaS)...')
cond_gen = ConditionGenerator(mode=CONDITION_MODE)

print('\nLoading BIM generator (ControlNet + SD 1.5)...')
bim_gen = BIMGenerator(condition_mode=CONDITION_MODE)

print('\nAll models ready!')


## Cell 9 — Single Sample Test

Verify the full pipeline end-to-end before running on the full dataset.

In [ ]:
import matplotlib.pyplot as plt

sample   = ds[0]
real_img = sample['image'].convert('RGB').resize((SIZE, SIZE))

print('Generating depth map...')
cond_map = cond_gen.generate(real_img)

pos, neg = build_prompt(sample)
print(f'Prompt: {pos}')

print('Generating BIM render...')
bim_img = bim_gen.generate(cond_map, pos, neg,
                            steps=STEPS, guidance=GUIDANCE,
                            cn_scale=CN_SCALE, size=SIZE)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(real_img);  axes[0].set_title('Real Photo',  fontsize=13); axes[0].axis('off')
axes[1].imshow(cond_map);  axes[1].set_title('Depth Map',   fontsize=13); axes[1].axis('off')
axes[2].imshow(bim_img);   axes[2].set_title('BIM Render',  fontsize=13); axes[2].axis('off')
plt.suptitle(f"view: {sample.get('view','')} | {sample.get('camera_distance','')}", y=1.02)
plt.tight_layout()
plt.show()
print('Single sample test passed!')


## Cell 10 — Full Dataset Generation

Processes all samples in `ds`. Already-generated images are skipped,
so the run is safely resumable after interruption.

Set `MAX_SAMPLES = None` in Cell 6 to process the full 7K dataset.

In [ ]:
import json
from tqdm.notebook import tqdm

META_PATH = os.path.join(OUT_BASE, 'metadata.jsonl')
meta_buf, errors, skipped = [], [], 0

for i, sample in enumerate(tqdm(ds, desc='BIM Generation')):
    image_id = sample.get('image_id', str(i).zfill(7))
    bim_out  = os.path.join(BIM_DIR, f'{image_id}.jpg')

    if os.path.exists(bim_out):  # resume support
        skipped += 1
        continue

    try:
        real_img = sample['image'].convert('RGB').resize((SIZE, SIZE))
        cond_map = cond_gen.generate(real_img)
        pos, neg = build_prompt(sample)
        bim_img  = bim_gen.generate(cond_map, pos, neg,
                                     steps=STEPS, guidance=GUIDANCE,
                                     cn_scale=CN_SCALE, size=SIZE)

        real_img.save(os.path.join(REAL_DIR, f'{image_id}.jpg'), quality=95)
        bim_img.save(bim_out, quality=95)
        cond_map.save(os.path.join(COND_DIR, f'{image_id}.jpg'), quality=95)

        meta_buf.append({
            'image_id':        image_id,
            'condition_mode':  CONDITION_MODE,
            'view':            sample.get('view', ''),
            'camera_distance': sample.get('camera_distance', ''),
            'illumination':    sample.get('illumination', ''),
            'caption':         sample.get('image_caption', '')[:200],
        })

        if len(meta_buf) >= 20:
            with open(META_PATH, 'a') as f:
                for r in meta_buf: f.write(json.dumps(r) + '\n')
            meta_buf = []

    except Exception as e:
        errors.append({'image_id': image_id, 'error': str(e)})
        print(f'  [{image_id}] ERROR: {e}')

if meta_buf:
    with open(META_PATH, 'a') as f:
        for r in meta_buf: f.write(json.dumps(r) + '\n')

n_success = len(list(os.scandir(BIM_DIR)))
print(f'Done! Generated: {n_success} | Skipped: {skipped} | Errors: {len(errors)}')
print(f'Output: {OUT_BASE}')


## Cell 11 — Visualize Results

In [ ]:
import glob, random, matplotlib.pyplot as plt
from PIL import Image as PILImage

real_files = sorted(glob.glob(os.path.join(REAL_DIR, '*.jpg')))

if not real_files:
    print('No generated images found. Run Cell 9 or Cell 10 first.')
else:
    n_show     = min(8, len(real_files))
    show_files = random.sample(real_files, n_show)

    fig, axes = plt.subplots(n_show, 3, figsize=(15, n_show * 4))
    if n_show == 1: axes = [axes]

    for ax, title in zip(axes[0], ['Real Photo', 'Depth Map', 'BIM Render']):
        ax.set_title(title, fontsize=13, fontweight='bold')

    for i, rp in enumerate(show_files):
        fname = os.path.basename(rp)
        axes[i][0].imshow(PILImage.open(rp))
        if os.path.exists(os.path.join(COND_DIR, fname)):
            axes[i][1].imshow(PILImage.open(os.path.join(COND_DIR, fname)))
        if os.path.exists(os.path.join(BIM_DIR, fname)):
            axes[i][2].imshow(PILImage.open(os.path.join(BIM_DIR, fname)))
        for ax in axes[i]: ax.axis('off')

    plt.suptitle(f'Results | mode={CONDITION_MODE} | guidance={GUIDANCE} | {n_show} samples',
                 fontsize=13, y=1.01)
    plt.tight_layout()
    grid_path = os.path.join(OUT_BASE, 'results_grid.png')
    plt.savefig(grid_path, dpi=100, bbox_inches='tight')
    plt.show()
    print(f'Grid saved: {grid_path}')


## Cell 12 — BIM Style Score

Measures how BIM-like the generated renders are.

**Formula:** low color saturation + limited hue variety = high score (flat colors = BIM aesthetic)

| Score | Quality |
|---|---|
| < 0.40 | Poor |
| 0.40 - 0.55 | Good |
| > 0.55 | Excellent |

**Baseline (50 samples): 0.61**

In [ ]:
import glob, numpy as np, matplotlib.pyplot as plt
from PIL import Image as PILImage

def bim_style_score(bim_path: str) -> float:
    img          = PILImage.open(bim_path).convert('HSV')
    arr          = np.array(img)
    saturation   = arr[:,:,1].mean() / 255.0
    unique_ratio = len(np.unique(arr[:,:,0])) / 256.0
    return float(1.0 - (saturation * 0.6 + unique_ratio * 0.4))

bim_files = glob.glob(os.path.join(BIM_DIR, '*.jpg'))

if not bim_files:
    print('No BIM renders found. Run Cell 10 first.')
else:
    scores = [bim_style_score(f) for f in bim_files]
    print(f'BIM Style Score over {len(scores)} samples')
    print(f'  Mean    : {np.mean(scores):.4f}  (target > 0.55)')
    print(f'  Std     : {np.std(scores):.4f}')
    print(f'  Min/Max : {np.min(scores):.4f} / {np.max(scores):.4f}')

    fig, ax = plt.subplots(figsize=(8, 3))
    ax.hist(scores, bins=20, color='steelblue', edgecolor='white')
    ax.axvline(np.mean(scores), color='red', linestyle='--',
               label=f'Mean: {np.mean(scores):.3f}')
    ax.set_xlabel('BIM Style Score')
    ax.set_ylabel('Frequency')
    ax.set_title(f'BIM Style Score Distribution ({len(scores)} samples)')
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_BASE, 'bim_style_score.png'), dpi=120)
    plt.show()


## Cell 13 — Push to GitHub

In [ ]:
import os, shutil

if 'REPO_DIR' not in dir():
    from google.colab import userdata
    try:
        GITHUB_REPO     = userdata.get('GITHUB_REPO')
        GITHUB_USERNAME = userdata.get('GITHUB_USERNAME')
    except:
        GITHUB_REPO     = input('Repo name: ').strip()
        GITHUB_USERNAME = input('GitHub username: ').strip()
    REPO_DIR = f'/content/{GITHUB_REPO}'

# Copy notebook to repo
for nb_name in ['BIM_Generation_Colab_v3.ipynb', 'BIM_Generation_Colab_v2.ipynb']:
    nb_src = f'/content/{nb_name}'
    if os.path.exists(nb_src):
        shutil.copy(nb_src, os.path.join(REPO_DIR, 'notebooks', nb_name))
        print(f'Notebook copied: {nb_name}')

# Copy 3 sample images into samples/ (outputs/ is gitignored, samples/ is not)
samples_dir = os.path.join(REPO_DIR, 'samples')
os.makedirs(samples_dir, exist_ok=True)
real_files = sorted(os.listdir(REAL_DIR))[:3] if os.path.exists(REAL_DIR) else []
for fname in real_files:
    for src_dir, prefix in [(REAL_DIR, 'real'), (BIM_DIR, 'bim')]:
        src = os.path.join(src_dir, fname)
        if os.path.exists(src):
            shutil.copy(src, os.path.join(samples_dir, f'{prefix}_{fname}'))
print(f'{len(real_files)} sample pairs copied to samples/')

# Update README
n_generated = len(list(os.scandir(BIM_DIR))) if os.path.exists(BIM_DIR) else 0
with open(os.path.join(REPO_DIR, 'README.md'), 'w') as f:
    f.write(f'# BIM Generation Pipeline\n\n'
            f'Real construction photos -> ControlNet -> BIM-style renders\n\n'
            f'## Results ({n_generated} renders generated)\n\n'
            f'| Metric | Value |\n|---|---|\n'
            f'| BIM Style Score | 0.61 (target > 0.55) |\n'
            f'| Condition mode  | depth (MiDaS) |\n'
            f'| Guidance scale  | 12.0 |\n'
            f'| ControlNet scale| 1.2 |\n'
            f'| Steps           | 35 |\n\n'
            f'## Next Steps\n\n'
            f'- [ ] YOLOv8 human removal from depth maps\n'
            f'- [ ] Full 7K dataset generation\n'
            f'- [ ] VLM fine-tuning on paired data\n')
print('README updated')

# Commit and push
os.chdir(REPO_DIR)
os.system('git add -A')
commit_msg = f'feat: BIM pipeline v3 | {n_generated} renders | JIT patch + style score'
ret = os.system(f'git commit -m "{commit_msg}"')
if ret == 0:
    push = os.system('git push origin main')
    if push == 0:
        print(f'Push complete!')
        print(f'  https://github.com/{GITHUB_USERNAME}/{GITHUB_REPO}')
    else:
        print('Push failed — check token permissions')
else:
    print('Nothing to commit')


## Next Steps

| Step | Description | Status |
|---|---|---|
| A | YOLOv8 human removal from depth maps | Next session |
| B | Set `MAX_SAMPLES = None` and generate full 7K dataset | After human removal |
| C | VLM fine-tuning on `(real_photo, BIM_render)` pairs | Final goal |

### Human Removal Plan
YOLOv8 detects person bounding boxes in the real photo. The corresponding regions
in the depth map are inpainted with surrounding values. ControlNet then no longer
sees human silhouettes, producing cleaner BIM renders without worker figures.